RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
###
def process_all_pdfs(pdf_directory):
    """Process all pdfs directly"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    # Find all pdfs
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\n Processing: {pdf_file.name} ")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata['file_type'] = "pdf"
            
            all_documents.extend(documents)
        
        except Exception as exception:
            print(f"Error {str(exception)}")
        
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_documents = process_all_pdfs("../data")

In [ ]:
###Textplitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap = 200):
    """Split documents into smaller chunks for better RAG Performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\n Example chunk:")
        print(f"Content {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

chunks = split_documents(all_documents)

Embedding and vectorStoreDB

In [4]:
import uuid
import chromadb
import numpy as np
from chromadb.config import Settings
from typing import List, Dict, Tuple, Any
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """
        Load the SentenceTransformer model
        """
        try:
            print(f"Loading Embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.get_embeddings_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def get_embeddings_dimension(self):
        """
        Get the embedding dimension of the model
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        return self.model.get_sentence_embedding_dimension()

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """"
        Generate embeddings for a list of Texts
        Args:
            text: List of text strings to embed
        
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generate embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated Embeddings with shape: {embeddings.shape}")
        return embeddings
    
## Initializing the embedding Manager

embedding_manager = EmbeddingManager()
embedding_manager

### Vector Store

In [ ]:
class VectorStore:
    """
        Manages document embedding in a ChromaDB Vector 
    """

    def __init__(self, collection_name:str = "pdf_documents", persist_directory:str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the Chromadb collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        

    def _initialize_store(self):
        """
            Initialize ChromaDB client and collection
        """

        try:
            #Creating persisten ChromaDb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
            print(f"Vector Store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            pass
        except Exception as exception:
            print(f"Error Initializing Vector Store..")
            raise

    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Adding documents and their embedding to the vector store
        
        Args:
            documents: List of langchain documents
            embeddings: Corresponding embedding for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings...")
        
        print(f"Adding {len(documents)} documents to vector store..")

        ids, metadatas , documents_text, embeddings_list = [], [] ,[] ,[]

        for index, (document, embedding) in enumerate(zip(documents, embeddings)):
            document_id = f"doc_{uuid.uuid4().hex[:8]}_{index}"
            ids.append(document_id)

            #prepare metadata
            metadata = dict(document.metadata)
            metadata['document_index'] = index
            metadata['content_length'] = len(document.page_content)
            metadatas.append(metadata)

            documents_text.append(document.page_content)
            embeddings_list.append(embedding.tolist())

            #Add to collection

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Sucessfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as exception:
            print(f"Error adding documents to vector store {exception}")
            raise
        

vectorstore = VectorStore()
vectorstore

In [ ]:
from langchain_core.documents import Document
### Convert the text to embedding
documents = chunks

### Generate the embeddings

embeddings = embedding_manager.generate_embeddings(
    texts=[document.page_content for document in documents])

##store in the vector database

vectorstore.add_documents(documents=documents, embeddings=embeddings)

### Retreival Pipeline From VectorStore 

In [8]:
class RagRetreival:
    """ Handles query-based retreival from the vector store """
    def __init__(self, vectore_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retreival

        Arg: 
            vectore_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vectore_store
        self.embedding_manager = embedding_manager
        
    
    def retrieve(self, query:str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
            Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of documents to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionarys containing retrieved documents and metadata
            
        """

        print(f"Retieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        #Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vectore store

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for index, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'metadata': metadata,
                            'content': document,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': index + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found..")

            return retrieved_docs
        except Exception as error:
            print(f"Error document retrieval: {error}")
            return []

rag_retrieval = RagRetreival(vectore_store=vectorstore,embedding_manager=embedding_manager)

In [ ]:
rag_retrieval.retrieve("What are embeddings?")

### Integration VectorDB Context Pipeline with LLM Output

In [19]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

#Initialize ChatOpenAI
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(api_key=OPENAI_API_KEY,model="gpt-4.1-mini",temperature=0.2, max_completion_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response

def rag_simple(query:str, retriever: RagRetreival, llm: ChatOpenAI, top_k:int=3):
    #retrieve context
    results = retriever.retrieve(query=query,top_k=top_k)

    context = "\n\n".join([document['content'] for document in results]) if results else ""
    if not context:
        return "Not relevant context found to answer the question.."
    
    prompt =f""" 
        You are a helpful assitant. Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    full_response = ""
    print("Response: ", end="", flush=True)
    for chunk in llm.stream(prompt.format(context=context, query = query)):
        print(chunk.content, end="", flush=True)
        full_response +=chunk.content
    # response = llm.invoke(prompt.format(context=context, query = query))
    return full_response


In [ ]:
answer = rag_simple("What is attention mechanism?",rag_retrieval, llm)
print(answer)

In [24]:
def rag_advance(query, retriever, llm, top_k=5, min_score=0.2, return_context = False):
    """
        RAG Pipeline with extra features:
        - Returns answer, sourcesm confidence score, and optionally full context
    """
    results = retriever.retrieve(query=query,top_k=top_k, score_threshold=min_score)
    if not results:
        return {"answer": 'No relevant context found.', 'sources': [], 'confidence':0.0,
                'context':''}

    #prepare context and sources
    context = "\n\n".join([document['content'] for document in results]) if results else ""

    sources = [{
        'source': document['metadata'].get('source_file', document['metadata'].get('source','unknown')),
        'page':document['metadata'].get('page','unknown'),
        'score':document['similarity_score'],
        'preview': f"{document['content'][:120]}..." 
    } for document in results]
    confidence = max([document['similarity_score'] for document in results])
    prompt =f""" 
    You are a helpful assitant. Use the following context to answer the question concisely.
    Context:
    {context}

    Question: {query}

    Answer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context
    
    return output
    

In [ ]:
result = rag_advance("What is attention mechanism?", retriever=rag_retrieval, llm=llm, top_k=3, min_score=0.1, return_context=True)

In [ ]:
result